 # Introduction to local search methods

 This notebook provides a simple implementation of **hill-clibing** and **local beam** search methods.

 **Important note:**

The Python code was created with the help of a generative AI assistant, so some fragments were drafted based on code taken from web.

### Common functions definition

In [ ]:
# Imports
import random

# Board representation: board[r] = c means the queen in row r is in column c.
# (One queen per row, so row conflicts are impossible by design.)

# Checks if it is safe to place a queen at the given position.
def is_safe(board, row, col):
  for i in range(row):                      # only rows above are already filled
    if board[i] == col or \
       abs(board[i] - col) == row - i:      # same column, or same diagonal (|Δcol| == |Δrow|)
      return False
  return True

# Calculates the number of conflicts in the current board.
# This is the heuristic h: 0 means a valid solution; lower is better.
def evaluate(board):
  conflicts = 0
  for i in range(len(board)):
    for j in range(i + 1, len(board)):      # each pair of queens counted once
      if board[i] == board[j] or \
         abs(board[i] - board[j]) == j - i: # same column, or same diagonal
        conflicts += 1
  return conflicts

### Hill-clibing search

In [ ]:
# Source: https://github.com/Mayur9167/AI_Puzzle_Solution

def hill_climbing(board_size):
  """Performs the hill climbing algorithm for the n-queens problem."""
  current_board = [random.randint(0, board_size - 1) for _ in range(board_size)]
  current_cost = evaluate(current_board)
  # print(current_board)

  while True:
    # Generate all neighbors: boards that differ by moving ONE queen within its row.
    # n rows x (n-1) other columns -> n*(n-1) neighbors (56 for n = 8).
    neighbors = []
    for i in range(board_size):             # pick the queen in row i
      for j in range(board_size):           # try every column j for it
        if current_board[i] != j:           # skip its current column
          neighbor_board = list(current_board)  # copy, so current_board stays unchanged
          neighbor_board[i] = j             # move queen i to column j
          neighbors.append(neighbor_board)

    best_neighbor = None
    best_neighbor_cost = float('inf')

    for neighbor in neighbors:
      neighbor_cost = evaluate(neighbor)
      if neighbor_cost < best_neighbor_cost:
        best_neighbor_cost = neighbor_cost
        best_neighbor = neighbor

    if best_neighbor is None or best_neighbor_cost >= current_cost:
      return current_board, current_cost

    current_board = best_neighbor
    current_cost = best_neighbor_cost

### Local beam search


In [ ]:
# Source:  https://github.com/Mayur9167/AI_Puzzle_Solution

def local_beam_search(board_size, beam_width):
  """Performs the local beam search algorithm for the n-queens problem."""
  # Start with k = beam_width random boards (instead of just one, as in hill climbing).
  current_boards = [[random.randint(0, board_size - 1) for _ in range(board_size)] for _ in range(beam_width)]
  current_costs = [evaluate(board) for board in current_boards]

  while True:
    # Generate the neighbors of ALL k boards and put them in one shared pool.
    # Each board has n*(n-1) neighbors -> k*n*(n-1) in total (30*56 = 1680 for k=30, n=8).
    
    neighbors = []
    for board in current_boards:
      for i in range(board_size):           # pick the queen in row i
        for j in range(board_size):         # try every column j for it
          if board[i] != j:                 # skip its current column
            neighbor_board = list(board)    # copy, so the original board stays unchanged
            neighbor_board[i] = j           # move queen i to column j
            neighbors.append(neighbor_board)

    neighbor_costs = [evaluate(neighbor) for neighbor in neighbors]

    # Keep only the k best neighbors from the whole pool (lowest conflicts first).
    # A promising board can contribute several successors, a bad one may contribute none.
    sorted_neighbors = sorted(zip(neighbors, neighbor_costs), key=lambda x: x[1])
    new_boards = [board for board, cost in sorted_neighbors[:beam_width]]
    new_costs = [cost for board, cost in sorted_neighbors[:beam_width]]

    # Goal test: the best new board has no conflicts -> solution found.
    if new_costs[0] == 0:
      return new_boards[0], new_costs[0]

    # No progress: the costs of the beam did not change -> stuck, return the best board.
    if new_costs == current_costs:
      return current_boards[0], current_costs[0]


    # The k selected boards become the new beam.
    current_boards = new_boards
    current_costs = new_costs

### Run the algorithm

In [ ]:
# Example usage for the 4-queens problem
board_size = 8
beam_width = 30

solution, cost = hill_climbing(board_size)
# solution, cost = local_beam_search(board_size, beam_width)

if cost == 0:
  print("Solution found:")
  evaluate(solution)
  for row in range(board_size):
    line = ""
    for col in range(board_size):
      if solution[row] == col:
        line += "Q "
      else:
        line += ". "
    print(line)
else:
  print("Local optimum found with", cost, "conflicts.")


---
## Exercises
1. Create a function that runs the an algorithm (e.g. hill_climbing) n times (default = 100) and counts the number of successes and failures (local otima).

2. Is local search applicable to finding a solution for the next problems?
  * 8 puzzle
  * Traveling salesman

In [ ]:
# prompt: create a function that takes an algorithm as a input (e.g. hill-clibing), runs it n times (default = 100) and returns the number of successes and failures.

def run_algorithm_multiple_times(algorithm, board_size, n=100, beam_width=None):
    """Runs the specified algorithm multiple times and counts successes/failures.

    Args:
        algorithm: The algorithm function to run (e.g., hill_climbing, local_beam_search).
        board_size: The size of the board for the n-queens problem.
        n: The number of times to run the algorithm (default: 100).
        beam_width: The beam width for local beam search (required if using that algorithm)

    Returns:
        A tuple containing the number of successes and failures.
    """
    successes = 0
    failures = 0

    for _ in range(n):
        if algorithm == hill_climbing:
            solution, cost = algorithm(board_size)
        elif algorithm == local_beam_search:
          if beam_width is None:
            raise ValueError("beam_width must be provided for local_beam_search")
          solution, cost = algorithm(board_size, beam_width)
        else:
          raise ValueError("Invalid algorithm specified")

        if cost == 0:
            successes += 1
        else:
            failures += 1

    return successes, failures

In [ ]:
import time
import tracemalloc

def measure(name, algorithm, board_size, n=100, beam_width=None):

  # 1st test
  # Time: total wall-clock time for n runs (tracemalloc is off, so it doesn't slow things down).
  start = time.perf_counter()
  successes, failures = run_algorithm_multiple_times(algorithm, board_size, n, beam_width)
  elapsed = time.perf_counter() - start


  # 2nd test
  # Memory: peak memory allocated by Python in each of n runs.
  # Runs start from random boards, so the peak varies -> report average and maximum.
  peaks = []
  tracemalloc.start()
  for _ in range(n):
    tracemalloc.reset_peak()                # start counting the peak again for this run
    baseline, _ = tracemalloc.get_traced_memory()  # memory already in use before the run
    if beam_width is None:
      algorithm(board_size)
    else:
      algorithm(board_size, beam_width)
    _, peak = tracemalloc.get_traced_memory()
    peaks.append((peak - baseline) / 1024)  # only what this run added, bytes -> KB
  tracemalloc.stop()

  print(f"{name}: Successes: {successes}, Failures: {failures}, "
        f"Time: {elapsed:.3f} s ({elapsed / n * 1000:.2f} ms/run), "
        f"Peak memory: avg {sum(peaks) / n:.1f} KB, max {max(peaks):.1f} KB")

measure("Local Beam Search", local_beam_search, 8, beam_width=30)
measure("Hill Climbing Search", hill_climbing, 8)